In [1]:
import pandas as pd
import numpy as np
import os
import sys
from os.path import join, abspath

parent_dir = abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from utils.db_helpers import execute_sql_select
from config import DATA_DIR


## Decoding

In [2]:
df_d = execute_sql_select("""SELECT * FROM decoding""")

Connection to DB closed


In [3]:
df_d.shape

(50331, 24)

In [4]:
df_d["code"].value_counts()

code
I0       39424
I0c       3973
I1ASC     3559
I1        3375
Name: count, dtype: int64

In [5]:
df_d[df_d["code"].isin(["I0","I1"])].shape

(42799, 24)

In [6]:
df_d[df_d["code"].isin(["I0","I1"])]["code"].value_counts(normalize=True)

code
I0    0.921143
I1    0.078857
Name: proportion, dtype: float64

In [7]:
df_d.comment_date_time = pd.to_datetime(df_d.comment_date_time)

In [8]:
df_d.comment_date_time.min()

Timestamp('2020-12-14 00:00:00')

In [9]:
df_d.comment_date_time.max()

Timestamp('2023-11-25 00:00:00')

In [10]:
df_d.source_outlet.nunique()

42

In [11]:
df_d.source_outlet.value_counts(normalize=True).head(5)

source_outlet
GUARD    0.123416
BBC      0.108142
DAILY    0.107481
TIMES    0.097732
INDEP    0.091786
Name: proportion, dtype: float64

In [12]:
df_d.source_type.value_counts(normalize=True)

source_type
Facebook    0.527528
YouTube     0.210884
Newspage    0.180326
Twitter     0.081262
Name: proportion, dtype: float64

## Bloomington

In [13]:
df_b = execute_sql_select("""SELECT * FROM bloomington""")

Connection to DB closed


In [14]:
df_b.shape

(11311, 10)

In [15]:
df_b["code"].value_counts()

code
0    9358
1    1953
Name: count, dtype: int64

In [16]:
df_b["code"].value_counts(normalize=True)

code
0    0.827336
1    0.172664
Name: proportion, dtype: float64

In [17]:
df_b.groupby("keyword")["code"].value_counts(normalize=True)

keyword  code
Israel   0       0.839381
         1       0.160619
Jews     0       0.886096
         1       0.113904
Kikes    0       0.657244
         1       0.342756
ZioNazi  1       0.882798
         0       0.117202
Name: proportion, dtype: float64

In [ ]:

bloomington_tmp = pd.read_feather(join(DATA_DIR, "bloomington_label_1.feather"))
column_name_renaming = {
    'classification_ihra_explanation_cleaned': 'IHRA',
    'classification_lexicon': 'LEXICON'
}

bloomington_tmp.rename(columns=column_name_renaming, inplace=True)

REL_CLASS_COLS = column_name_renaming.values()

In [ ]:
# percentage of annotations were given the same IHRA sections by both annotators
100*(1 - np.round(bloomington_tmp["ihra_sections"].map(lambda x: len(x)>1).sum()/bloomington_tmp.shape[0],2))

In [ ]:
# Create a new df for Bloomington data with a new column "ihra_sec_new" containing both annotators' decisions.
bloomington_copy_x = bloomington_tmp.copy()
bloomington_copy_y = bloomington_tmp.copy()
bloomington_copy_x["ihra_sec_new"] = bloomington_tmp["ihra_section_1"]
bloomington_copy_y["ihra_sec_new"] = bloomington_tmp["ihra_section_2"]
bloomington = pd.concat([bloomington_copy_x, bloomington_copy_y], ignore_index=True)
print(len(bloomington))
bloomington = bloomington[bloomington["ihra_sec_new"]!=13] # there are a few errors coded as 13
print(len(bloomington))

In [ ]:
bloomington.ihra_sec_new.value_counts(normalize=True).cumsum() 

In [ ]:
for k in bloomington.keyword.unique():
    print(f"Keyword: {k}")
    df_k = bloomington[bloomington.keyword==k]
    print(len(df_k))
    print(df_k.ihra_sec_new.value_counts(normalize=True).cumsum())

**Top IHRA sections per keyword sample**

- keyword `israel`: 7 and 9 make up 85%
- keyword `kikes`: 0 and 2 make up 90%
- keyword `zionazi`: 10, 2 and 0 make up 88% (with 7 we obtain 95%)
- keyword `jews`: needs 5 codes to achieve a similar coverage (>=87%) but 0,7,2 are top 3 and cover 77%